# 上下文压缩

上下文压缩是指通过技术手段精简上下文内容，同时保证语义连贯。

上下文压缩的介入点主要有两个：

- **新消息防爆**：需要立即处理的新消息
- **上下文摘要**：需要处理的旧消息

## 新消息防爆

> 新消息防爆的处理特点是：
>
> 1. 针对的是新产生的消息
> 2. 处理是立即发生的
>
> 新消息代表着目前模型正在工作的内容，处理起来要非常谨慎，绝不能打破消息之间的因果关系。

### 工具返回防爆

主要手段是：**窗口控量，卸载兜底**

#### 窗口控量

那些容易造成大结果的工具，需要通过参数让模型不要一次性拿完整结果。

比如`read_file`工具提供`offset`和`limit`，并给予了参数默认值

#### 卸载兜底

若工具返回超过了阈值，直接卸载。

```
[卸载后的结果] + [完整结果引用] → 模型
```



> 卸载兜底功能已由`FilesystemMiddleware`中间件实现。
>
> 你可以配置工具的卸载阈值
>
> ```python
> FilesystemMiddleware(
>   tool_token_limit_before_evict=20000
> )
> ```


> 你可以配置特殊
>
> ```python
> backend=CompositeBackend(
>   default=...,
>   routes={},
>   # 可配置 Deep Agent 内部产物的保存根目录
>   artifacts_root="/home/user/workspace/.deepagents",
> )
> ```



### 用户消息防爆

用户消息超过阈值，直接卸载

> 卸载兜底功能已由`FilesystemMiddleware`中间件实现。
>
> 你可以配置用户消息的卸载阈值
>
> ```python
> FilesystemMiddleware(
> 	human_message_token_limit_before_evict=50000
> )
> ```


In [3]:
from langgraph_sdk import get_client

client = get_client(url="http://127.0.0.1:2024")

assistant_id = 'bc46c028-583a-563c-8327-8ab22872af90'
# 创建新线程
thread = await client.threads.create(metadata={"__name__": "用户消息卸载"})
thread_id = thread["thread_id"]
thread_id, assistant_id

('01a05bd2-25aa-70d2-acef-7e8a51bf8382',
 'bc46c028-583a-563c-8327-8ab22872af90')

In [ ]:
resp = await client.runs.wait(
    input={
        "messages":[
            {
                "role":"user",
                "content": "\n".join([f"第{i}条消息" for i in range(1, 100000)])
            }
        ]
    },
    thread_id=thread_id, 
    assistant_id=assistant_id
)

> 思考：
>
> 1. 系统提示词过大要不要立即处理？
> 2. 工具参数过大要不要立即处理？
> 3. AI回复的普通消息要不要立即处理？

## 上下文摘要

上下文摘要只处理旧消息，它会让旧的因果关系变得模糊。

但由于它是旧的因果关系，因此这种模糊对整个会话的影响较小。

### 处理旧的特殊工具参数

在Agent运行期间，有两个特殊工具调用：

- `write_file`：传入的新内容可能过大
- `edit_file`：传入的旧内容和新内容都可能过大

这两个工具都有共同特点：

- 工具的执行后，有价值的结果已落入文件
- 模型往往关心的是文件当下的状态，而工具的调用历史只代表改动过程
- 工具调用和工具返回的结果本身的因果关系就是暗淡的

```python
summarization_middleware = SummarizationMiddleware(
    model=model,
    backend=backend,

    # 旧工具调用参数卸载配置
    truncate_args_settings={
        # 对话达到模型上下文窗口的 85% 时，开始检查旧工具调用参数
        "trigger": ("fraction", 0.85),

        # 最近 10% 的上下文中的工具参数保持完整
        "keep": ("fraction", 0.10),

        # 单个参数字符串超过 2000 个字符才卸载
        "max_length": 2_000,

        # 卸载后的提示文字
        "truncation_text": "...(argument truncated)",
    },
)
```

### 处理整个上下文中的旧消息

```python
summarization_middleware = SummarizationMiddleware(
    model=model,
    backend=backend,

    # 整个对话达到模型上下文窗口的 85% 时触发摘要
    trigger=("fraction", 0.85),

    # 摘要时保留最近 10% 的上下文不参与摘要
    keep=("fraction", 0.10),
)
```

In [4]:
assistant_id = '58534c12-9bc2-5cd2-b195-be42dfcd1f1f'
# 创建新线程
thread = await client.threads.create(metadata={"__name__": "历史上下文摘要"})
thread_id = thread["thread_id"]
thread_id, assistant_id

('01a05bea-22bc-79f0-8ff6-c40e98bdbc46',
 '58534c12-9bc2-5cd2-b195-be42dfcd1f1f')

In [ ]:
messages = []

# 构造约 8.6 万 tokens 的历史对话：超过 85% 的摘要阈值，但摘要后只保留最近 10%
for round_id in range(1, 11):
    user_context = "\n".join(
        f"第{round_id}轮需求记录{i}：报表需保留订单编号、客户名称、销售额和退款状态。"
        for i in range(1, 426)
    )
    assistant_context = "\n".join(
        f"第{round_id}轮处理记录{i}：已核对字段映射、异常值规则和汇总口径。"
        for i in range(1, 426)
    )
    messages.extend([
        {"role": "user", "content": user_context},
        {"role": "assistant", "content": assistant_context},
    ])


messages

[{'role': 'user',
  'content': '第1轮需求记录1：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录2：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录3：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录4：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录5：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录6：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录7：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录8：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录9：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录10：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录11：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录12：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录13：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录14：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录15：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录16：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录17：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录18：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录19：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录20：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录21：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录22：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录23：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录24：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录25：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录26：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求记录27：报表需保留订单编号、客户名称、销售额和退款状态。\n第1轮需求

In [ ]:
resp = await client.runs.wait(
    input={
        "messages":messages
    },
    thread_id=thread_id, 
    assistant_id=assistant_id
)